In [8]:
# 데이터 로딩 이전 단계(환경 확인)
import os

os.listdir('../data') 
# './' = 현재 작업 경로를 뜻함
# 이 안에 들어 있는 파일/폴더 이름을 list 형태로 반환

['sample_submission.csv', 'building_info.csv', 'train.csv', 'test.csv']

In [9]:
import pandas as pd 
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

import math
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 및 마이너스
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
submission = pd.read_csv('../data/sample_submission.csv')

print(train_df.shape, test_df.shape, submission.shape)

(204000, 10) (16800, 7) (16800, 2)


In [ ]:
train_df.info()

In [ ]:
# object 컬럼 제외
hist_df = train_df.drop(columns=['num_date_time','일시',], errors='ignore')

n_features = list(hist_df.columns)
n_cols = 5
n_rows = math.ceil(len(n_features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
ax = axes.flatten()

for i, col in enumerate(hist_df.columns):
    sns.histplot(hist_df[col].dropna(), bins=30, kde=True, ax=ax[i])
    ax[i].set_title(f'Distribution of {col}', fontsize=12)

for i in range(len(n_features), len(ax)):
    ax[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 일사/일조랑 상관이 큰 변수 확인

corr = train_df.corr(numeric_only=True)
target_corr = corr["일조(hr)"].drop(["전력소비량(kWh)", "일조(hr)", "일사(MJ/m2)"]).sort_values(ascending=False)

plt.figure(figsize=(6,8))
sns.barplot(
    x=target_corr.values, 
    y=target_corr.index, 
    palette="coolwarm"
)
plt.title("Target vs Features Correlation (excluding target)", fontsize=14)
plt.show()


target_corr = corr["일사(MJ/m2)"].drop(["전력소비량(kWh)", "일사(MJ/m2)", "일조(hr)"]).sort_values(ascending=False)

plt.figure(figsize=(6,8))
sns.barplot(
    x=target_corr.values, 
    y=target_corr.index, 
    palette="coolwarm"
)
plt.title("Target vs Features Correlation (excluding target)", fontsize=14)
plt.show()

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# 1) 날짜 파생변수 생성
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if '일시' in out.columns:
        if not np.issubdtype(out['일시'].dtype, np.datetime64):
            out['일시'] = pd.to_datetime(out['일시'], errors='coerce')
        out['year'] = out['일시'].dt.year
        out['month'] = out['일시'].dt.month
        out['day'] = out['일시'].dt.day
        out['hour'] = out['일시'].dt.hour
        out['dayofweek'] = out['일시'].dt.dayofweek
        out['quarter'] = out['일시'].dt.quarter
        out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)

        def season(m):
            if m in [12, 1, 2]:
                return 0
            if m in [3, 4, 5]:
                return 1
            if m in [6, 7, 8]:
                return 2
            return 3
        out['season'] = out['month'].apply(season)
    return out


In [10]:
train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

train_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204000 entries, 0 to 203999
Data columns (total 18 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   num_date_time  204000 non-null  object        
 1   건물번호           204000 non-null  int64         
 2   일시             204000 non-null  datetime64[ns]
 3   기온(°C)         204000 non-null  float64       
 4   강수량(mm)        204000 non-null  float64       
 5   풍속(m/s)        204000 non-null  float64       
 6   습도(%)          204000 non-null  float64       
 7   일조(hr)         204000 non-null  float64       
 8   일사(MJ/m2)      204000 non-null  float64       
 9   전력소비량(kWh)     204000 non-null  float64       
 10  year           204000 non-null  int32         
 11  month          204000 non-null  int32         
 12  day            204000 non-null  int32         
 13  hour           204000 non-null  int32         
 14  dayofweek      204000 non-null  int32         
 15  

In [ ]:
# object onehotencoding
train_df['건물번호'] = OneHotEncoder(sparse_output=False).fit_transform(train_df[['건물번호']])
test_df['건물번호'] = OneHotEncoder(sparse_output=False).fit_transform(test_df[['건물번호']])

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
203995    1.0
203996    1.0
203997    1.0
203998    1.0
203999    1.0
Name: 건물번호, Length: 204000, dtype: float64

In [ ]:
# 6) 결측치 채우기
def predict_for_df(model, df_like: pd.DataFrame):
    drop_targets = ['일사(MJ/m2)', '일조(hr)']
    drop_others = ['num_date_time']
    drop_cols = set(drop_targets + drop_others)

    base_cols = [c for c in df_like.columns if c not in drop_cols]
    X_all = add_time_features(df_like[base_cols])
    if '일시' in X_all.columns:
        X_all = X_all.drop(columns=['일시'])
    if '건물번호' in X_all.columns:
        X_all['건물번호'] = X_all['건물번호'].astype('category')
    return model.predict(X_all)

filled_df = train_df.copy()

# (a) 일사 결측 채우기
mask_irr_na = filled_df['일사(MJ/m2)'].isna()
if mask_irr_na.any():
    filled_df.loc[mask_irr_na, '일사(MJ/m2)'] = predict_for_df(model_irr, filled_df[mask_irr_na])

# (b) 일조 결측 채우기
mask_sun_na = filled_df['일조(hr)'].isna()
if mask_sun_na.any():
    filled_df.loc[mask_sun_na, '일조(hr)'] = predict_for_df(model_sun, filled_df[mask_sun_na])

# 7) 결과 확인
print("결측 보정 전/후 결측 개수")
print({
    '일사_before_na': train_df['일사(MJ/m2)'].isna().sum(),
    '일사_after_na':  filled_df['일사(MJ/m2)'].isna().sum(),
    '일조_before_na': train_df['일조(hr)'].isna().sum(),
    '일조_after_na':  filled_df['일조(hr)'].isna().sum(),
})
# 이후 파이프라인에 filled_df 사용
# train_df = filled_df
